In [43]:
from src.model_learned_tikhonov import LearnedTikhonov
from src.print_utils import print_kv

import torch
import h5py
import matplotlib.pyplot as plt

import numpy as np
import torch
import torch.nn as nn

from typing import Literal, get_args

plt.style.use("styles/plots.mplstyle")

In [44]:
def smoothness_regularization(model: LearnedTikhonov) -> torch.Tensor:
    """
    Penalizza variazioni brusche di R(omega) e Phi(omega).
    Assumendo che model.R e model.Phi siano nn.Parameter di shape (N_omega,).
    """
    dR   = model._get_R()[1:]   - model._get_R()[:-1]
    dPhi = model._get_phi()[1:] - model._get_phi()[:-1]
    return (dR ** 2).mean() + (dPhi ** 2).mean()

_TYPES = Literal["ReduceLR", "Cosine"]

def train(
    model: LearnedTikhonov,
    G_train: torch.Tensor,
    F_train: torch.Tensor,
    G_val:   torch.Tensor,
    F_val:   torch.Tensor,
    n_epochs: int        = 2000,
    lr: float            = 1e-3,
    weight_decay: float  = 0.0,
    checkpoint_path: str = "assets/tikhonov_learned/best_tikhonov.pt",
    scheduler_patience: int   = 50,
    scheduler_factor:   float = 0.5,
    log_every: int       = 50,
    lambda_smooth: float = 0,
    scheduler_type:_TYPES = "Cosine"
) -> dict: 

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    
    # Choose scheduler type
    options = get_args(_TYPES)
    assert scheduler_type in options, f"'{scheduler_type}' is not in {options}"
            
    if scheduler_type == "Cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=2000, T_mult=2, eta_min=1e-6
        )
    elif scheduler_type == "ReduceLR":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", patience=scheduler_patience,
            factor=scheduler_factor
        )

    loss_fn = nn.MSELoss()

    history = {"train_loss": [], "val_loss": [], "lr": []}
    best_val = float("inf")

    for epoch in range(1, n_epochs + 1):

        # ---- Training Step ----
        model.train()
        optimizer.zero_grad()
        
        pred_train = model(G_train)
        mse_train = loss_fn(pred_train, F_train)
        loss_train  = mse_train
        
        if lambda_smooth != 0:
            reg_smooth  = smoothness_regularization(model)
            loss_train  += lambda_smooth * reg_smooth
        
        
        loss_train.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # ---- Validation Step ----
        model.eval()
        with torch.no_grad():
            
            pred_val = model(G_val)
            mse_val = loss_fn(pred_val, F_val)
            loss_val = mse_val
            
            if lambda_smooth != 0:
                reg_smooth  = smoothness_regularization(model)
                loss_val  += lambda_smooth * reg_smooth
            

        train_l = loss_train.item()
        val_l   = loss_val.item()
        scheduler.step(val_l)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_l)
        history["val_loss"].append(val_l)
        history["lr"].append(current_lr)

        if val_l < best_val:
            best_val = val_l
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": best_val,
            }, checkpoint_path)

        if epoch % log_every == 0 or epoch == 1:
            print(f"Epoch {epoch:5d}/{n_epochs} | "
                  f"train MSE: {train_l:.4e} | "
                  f"val MSE: {val_l:.4e} | "
                  f"lr: {current_lr:.2e}")

    print(f"\nBest val MSE: {best_val:.4e}  ->  {checkpoint_path}")
    return history

In [45]:
def plot_diagnostics(model: LearnedTikhonov, freqs: np.ndarray,
                     save_path: str = "diagnostics.pdf"):
    fig, axes = plt.subplots(3, 1, figsize=(4.4, 6), sharex=True)

    ax1, ax2, ax3 = axes
    
    # Phi Plot
    ax1.plot(freqs, np.degrees(model.get_phi()), color="steelblue", lw=1.5)
    ax1.axhline(0, color="k", lw=0.5)
    ax1.set_ylabel(r"$\phi$")
    ax1.set_title("")
    ax1.grid(True, alpha=0.3)

    # R plot
    ax2.semilogy(freqs, model.get_R(), color="darkorange", lw=1.5)
    ax2.set_ylabel(r"$R$")
    ax2.set_title("")
    ax2.grid(True, alpha=0.3)

    # W plot
    ax3.semilogy(freqs, model.get_W_magnitude(), color="seagreen", lw=1.5)
    ax3.set_ylabel(r"$|W|$")
    ax3.set_xlabel(r"$\omega$")
    ax3.set_title("")
    ax3.grid(True, alpha=0.3)

    plt.savefig(save_path)
    plt.close()
    print(f"Diagnostic saved -> {save_path}")
    
def plot_history(history: dict, save_path: str = "loss_history.pdf"):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots()
    
    ax.semilogy(epochs, history["train_loss"], label="Train MSE")
    ax.semilogy(epochs, history["val_loss"],   label="Val MSE", ls="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.set_title("Loss curve")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig(save_path)
    plt.close()
    print(f"Loss history saved -> {save_path}")

def plot_prediction(model: LearnedTikhonov,
                    G_val: torch.Tensor,
                    F_val: torch.Tensor,
                    dt: float,
                    n_samples: int = 3,
                    save_path: str = "predictions.pdf"):

    model.eval()
    with torch.no_grad():
        f_pred = model(G_val).cpu().numpy()
    f_true = F_val.cpu().numpy()
    nt = f_true.shape[1]
    t  = np.arange(nt) * dt

    n_show = min(n_samples, len(f_true))
    fig, axes = plt.subplots(n_show, 1, figsize=(4.4, 1.2 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]

    G_array = G_val.cpu().numpy()
    for i, ax in enumerate(axes):
        
        ax.plot(t, f_true[i], label=r"$f_{true}$", lw=1.5)
        ax.plot(t, f_pred[i], label=r"$f_{pred}$", lw=1.5)
        ax.plot(t, G_array[i], label="g", lw=1.5)
        
        mse_i = np.mean((f_pred[i] - f_true[i]) ** 2)
        
        ax.text(
            0.02, 0.95,
            f"MSE = {mse_i:.3e}",
            transform=ax.transAxes,
            verticalalignment='top'
        )
        ax.set_ylabel("f(t)")
        ax.grid(True, alpha=0.3)
        ax.set_ylim([-3,3])

    axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("Time [s]")
    
    plt.savefig(save_path)
    plt.close()
    print(f"Predictions saved -> {save_path}")


In [46]:
# Load simulation parameters
L, Tf, nt, D, h_param, l, _, r, _ = np.loadtxt(
    "dataset/simulation_info.csv", delimiter=";", usecols=1
)
nt = int(nt)
dt = Tf / nt
n  = int(2 * nt - 1)

# Load impulse response
h_vals = np.loadtxt("dataset/impulse_response.csv", delimiter=";")
H_np   = np.fft.rfft(h_vals, n).astype(np.complex64)
nk     = len(H_np)
freqs  = np.fft.rfftfreq(n, d=dt)

print(f"nt={nt}, n={n}, nk={nk}, dt={dt:.4e}")

# Load dataset
VAL_FRACTION = 0.1
SEED         = 42
H5_PATH      = "dataset/dataset.h5"

with h5py.File(H5_PATH, "r") as hf:
    data = hf["dataset"][:]              # (N, 2, nt)

f_all = data[:, 0, :].astype(np.float32)
g_all = data[:, 1, :].astype(np.float32)


rolling = 0 # Simulate a delay of: tau = rolling * dt
if rolling != 0:
    g_rolled = np.roll(g_all, rolling)
    g_rolled[:,:rolling] = 0
    g_all = g_rolled

N = len(f_all)

rng       = np.random.default_rng(SEED)
idx       = rng.permutation(N)
n_val     = max(1, int(VAL_FRACTION * N))
idx_val   = idx[:n_val]
idx_train = idx[n_val:]

print(f"Dataset: tatl={N}, train={len(idx_train)}, val={n_val}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

G_train = torch.from_numpy(g_all[idx_train]).to(device)
F_train = torch.from_numpy(f_all[idx_train]).to(device)
G_val   = torch.from_numpy(g_all[idx_val]).to(device)
F_val   = torch.from_numpy(f_all[idx_val]).to(device)

# Init model

model = LearnedTikhonov(H=H_np, n=n, nt=nt, dt=dt).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parametri appresi: {n_params}  (phi: {nk}, log_R: {nk})")
history = None

nt=2000, n=3999, nk=2000, dt=5.0000e-03
Dataset: tatl=800, train=720, val=80
Device: cuda
Parametri appresi: 4000  (phi: 2000, log_R: 2000)


# 4. Training

In [47]:
perform_training = False

if perform_training:
    history = train(
        model        = model,
        G_train      = G_train,
        F_train      = F_train,
        G_val        = G_val,
        F_val        = F_val,
        n_epochs     = 15000,
        lr           = 1e-3,
        weight_decay = 0.0,
        checkpoint_path    = "assets/tikhonov_learned/best_tikhonov.pt",
        scheduler_patience = 50,
        scheduler_factor   = 0.5,
        log_every    = 100,
    )

# 5. Diagnostica

In [48]:
# Load trained model on CPU
ckpt = torch.load("assets/tikhonov_learned/best_tikhonov.pt", map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.to("cpu")

# Plot R, Phi, W
plot_diagnostics(model, 
                freqs, 
                save_path="assets/tikhonov_learned/diagnostics.pdf")

# Plot loss history
if history is not None:
    plot_history(history, 
                save_path="assets/tikhonov_learned/loss_history.pdf")
    
# Plot prediction of subset of validation_set
plot_prediction(model, G_val.cpu(), F_val.cpu(), dt,
                n_samples=min(20, n_val),
                save_path="assets/tikhonov_learned/predictions.pdf")

Diagnostic saved -> assets/tikhonov_learned/diagnostics.pdf
Predictions saved -> assets/tikhonov_learned/predictions.pdf


In [49]:
from src.heat1D_crank_nicolson_solver import Heat1DCrankNicolson
from src.tikhonov_handler import TikhonovHandler

# Initialize solver CrankNicholson with parameters loaded from simulation_info.csv
solver = Heat1DCrankNicolson(L=L,Tf=Tf,nt=nt,D=D,h=h_param,wanted_r=r)
solver.info()

th = TikhonovHandler(solver)

G_val_np = G_val.cpu().numpy()
F_true = F_val.cpu().numpy()
F_tikh_val = np.ndarray(shape=G_val_np.shape)

n_show = 20 # Number of example to show

i = 0
for G_val_i in G_val_np:
    F_tikh_val[i] = th.regularize(G_val_i, lambda_reg=1e-3, regularization_type_="closed",)
    i += 1

fig, axes = plt.subplots(n_show, 1, figsize=(10, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(solver.t_array, F_true[i], label="f true", lw=1.5)
    ax.plot(solver.t_array,  F_tikh_val[i], label="f pred", lw=1.5)
    ax.plot(solver.t_array,  G_val_np[i], label="g", lw=1.5)
    mse_i = np.mean((F_tikh_val[i] - F_true[i]) ** 2)
    ax.set_title(f"Campione val {i}  —  MSE = {mse_i:.3e}")
    ax.set_ylabel("f(t)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-3,3])

axes[-1].set_xlabel("Tempo [s]")
plt.savefig("assets/tikhonov_learned/predictions_tikh.pdf", dpi=150, bbox_inches="tight")
plt.close()

# Compute Tikhonov reconstruction MSE on validation set 
F_tikh_val = torch.tensor(F_tikh_val, dtype=F_val.dtype)
criterion = nn.MSELoss()
loss = criterion(F_tikh_val, F_val.cpu())

print_kv("val MSE tikh:", f"{loss:.3e}")


-----------------------------Solver info------------------------------

Diffusion time:                                               1.00e+00
dx:                                                           8.33e-02
dt:                                                           5.00e-03
nt:                                                               2000
nx:                                                                 13
r:                                                            7.20e-01
h:                                                            1.00e+00
val MSE tikh:                                                2.509e+02
